# Build your first AI agent! — Workshop notebook

Companion notebook for the Google ADK (Agent Development Kit) Python workshop.

**What's inside:** every workshop block in a single notebook — run them independently.

> ⚠️ This notebook version uses `InMemoryRunner` and `await` directly — that works in Colab/Jupyter where the event loop is already running. For a local run via `adk web`, see the `checkpoints/` folders.


## Block 2. Environment setup

First install ADK and wire up the Gemini API key.


In [1]:
# Uncomment on first run in Colab.
# google-adk[eval] also pulls pandas/tabulate/rouge — needed by Block 9.
!pip install -q 'google-adk[eval]' python-dotenv


zsh:1: command not found: pip


In [ ]:
import os

# Paste your key from https://aistudio.google.com/apikey
os.environ["GOOGLE_API_KEY"] = "PASTE_YOUR_KEY_HERE"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

# Sanity check
import google.adk
print(f"ADK version: {google.adk.__version__}")


ADK version: 2.1.0


## Block 3. First agent: Hello, Agent!

The minimal working agent — just LLM + instruction, no tools.


In [5]:
from google.adk.agents import Agent

root_agent = Agent(
    name="python_helper",
    model="gemini-2.5-flash",
    description="Friendly Python assistant.",
    instruction=(
        "You are a friendly Python mentor. "
        "Answer concisely, to the point, with code examples."
    ),
)
print(root_agent)


name='python_helper' description='Friendly Python assistant.' rerun_on_resume=False wait_for_output=False retry_config=None timeout=None input_schema=None output_schema=None state_schema=None parent_agent=None sub_agents=[] before_agent_callback=None after_agent_callback=None model='gemini-2.5-flash' instruction='You are a friendly Python mentor. Answer concisely, to the point, with code examples.' global_instruction='' static_instruction=None tools=[] generate_content_config=None mode=None parallel_worker=None disallow_transfer_to_parent=False disallow_transfer_to_peers=False include_contents='default' output_key=None planner=None code_executor=None before_model_callback=None after_model_callback=None on_model_error_callback=None before_tool_callback=None after_tool_callback=None on_tool_error_callback=None


### Run the agent via InMemoryRunner

Create a runner and session, send a message.


In [6]:
from google.adk.runners import InMemoryRunner
from google.genai import types

APP_NAME = "workshop"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)
print(f"Created session: {session.id}")


Created session: 7cac2a46-34c2-45db-9d7e-90e55660a08a


In [7]:
async def ask(text: str, session_id: str = session.id):
    """Helper: send a message and print the final response."""
    print(f">>> {text}")
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)]),
    ):
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"<<< {part.text.strip()}")

await ask("What is a list comprehension in Python? Show a short example.")


>>> What is a list comprehension in Python? Show a short example.
<<< A list comprehension provides a concise way to create lists. It's a syntactic sugar for a `for` loop combined with `list.append()`, often making the code more readable and efficient.

It generally follows the structure: `[expression for item in iterable if condition]`

**Example:**

```python
# Create a list of squares for numbers 0 to 4
squares = [x**2 for x in range(5)]

print(squares) # Output: [0, 1, 4, 9, 16]
```


## Block 4. Tools

Add function tools. ADK will turn them into tool schemas — the key thing is to have **type hints and docstrings**.


In [8]:
import datetime
from zoneinfo import ZoneInfo


def get_weather(city: str) -> dict:
    """Returns the current weather in the specified city.

    Args:
        city: City name in English (e.g., "New York").
    """
    fake_data = {
        "new york": "It's sunny in New York, 25°C.",
        "london": "It's cloudy in London, 14°C.",
        "tashkent": "It's clear in Tashkent, 28°C.",
    }
    report = fake_data.get(city.lower())
    if report:
        return {"status": "success", "report": report}
    return {"status": "error", "error_message": f"No data for {city}."}


def get_current_time(city: str) -> dict:
    """Returns the current time in the specified city."""
    tz_map = {
        "new york": "America/New_York",
        "london": "Europe/London",
        "tashkent": "Asia/Tashkent",
    }
    tz_id = tz_map.get(city.lower())
    if not tz_id:
        return {"status": "error", "error_message": f"No timezone for {city}."}
    now = datetime.datetime.now(ZoneInfo(tz_id))
    return {"status": "success", "report": now.strftime("%Y-%m-%d %H:%M:%S %Z")}


weather_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.5-flash",
    description="Agent that answers questions about weather and time in a city.",
    instruction=(
        "You are a helpful assistant. When the user asks about weather "
        "or time — you MUST use the tools."
    ),
    tools=[get_weather, get_current_time],
)

# Restart the runner with the new agent
runner = InMemoryRunner(agent=weather_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)
await ask("What's the weather and time in New York?", session.id)


>>> What's the weather and time in New York?


/Users/muhammad/Repos/AgentAI/ADK-workshop/.venv/lib/python3.14/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


<<< It's sunny in New York, 25°C. The current time is 2026-05-26 01:40:27 EDT.


## Block 5. Memory & Sessions

Three layers:
- **Session** — message history of a single conversation (stored automatically).
- **State** — structured data within a session (`tool_context.state`).
- **Memory** — long-term memory across sessions (`MemoryService`).


In [9]:
from google.adk.tools.tool_context import ToolContext


def remember_preference(key: str, value: str, tool_context: ToolContext) -> dict:
    """Saves a user preference to session state.

    Args:
        key: Key (e.g., "favorite_city").
        value: Value.
    """
    tool_context.state[key] = value
    return {"status": "success", "saved": {key: value}}


def recall_preference(key: str, tool_context: ToolContext) -> dict:
    """Retrieves a saved user preference."""
    value = tool_context.state.get(key)
    if value is None:
        return {"status": "error", "error_message": f"No value for {key}."}
    return {"status": "success", "value": value}


memory_agent = Agent(
    name="weather_time_agent_with_memory",
    model="gemini-2.5-flash",
    description="Agent that remembers user preferences.",
    instruction=(
        "You are a helpful assistant. Use remember_preference when the user "
        "states a preference, and recall_preference when you need to recall it."
    ),
    tools=[get_weather, get_current_time, remember_preference, recall_preference],
)

runner = InMemoryRunner(agent=memory_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

await ask("My name is Muhammad, my favorite city is Tashkent. Remember that.", session.id)


>>> My name is Muhammad, my favorite city is Tashkent. Remember that.
<<< I'll remember that your name is Muhammad and your favorite city is Tashkent.


In [12]:
# Same session_id — the agent remembers context
await ask("What's the weather in my favorite city?", session.id)


>>> What's the weather in my favorite city?
<<< It's clear in Tashkent, 28°C.


In [13]:
# Inspect what's in state
session_now = await runner.session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=session.id
)
print("Session state:", dict(session_now.state))


Session state: {'name': 'Muhammad', 'favorite city': 'Tashkent'}


## Block 6. Streaming

Turn on SSE streaming via `RunConfig` — the response starts arriving token-by-token.


In [14]:
from google.adk.agents.run_config import RunConfig, StreamingMode

storyteller = Agent(
    name="storytelling_agent",
    model="gemini-2.5-flash",
    description="Long-story storyteller.",
    instruction="When asked, tell long, detailed stories.",
)

runner = InMemoryRunner(agent=storyteller, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

run_config = RunConfig(
    streaming_mode=StreamingMode.SSE,
    max_llm_calls=20,
)

print(">>> Tell me a long story about a coding cat.\n")
print("<<< ", end="", flush=True)

async for event in runner.run_async(
    user_id=USER_ID,
    session_id=session.id,
    new_message=types.Content(
        role="user",
        parts=[types.Part(text="Tell me a long story about a coding cat.")],
    ),
    run_config=run_config,
):
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text, end="", flush=True)
    if event.is_final_response():
        print()


>>> Tell me a long story about a coding cat.

<<< The air in the apartment was thick with the scent of lukewarm coffee and the quiet hum of a dozen electronic devices. On a plush velvet cushion, nestled beside a stack of weighty textbooks on quantum mechanics, lay Miso, a sleek ginger tabby with unusually bright, intelligent emerald eyes. To the casual observer – specifically, his

/Users/muhammad/Repos/AgentAI/ADK-workshop/.venv/lib/python3.14/site-packages/google/adk/models/google_llm.py:260: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for llm_response in aggregator_gen:


 human, Oliver, a perpetually coding software engineer – Miso was just a very fluffy, very spoiled housecat. He chased laser pointers, napped in sunbeams, and demanded breakfast with the insistence of a tiny, furry tyrant.

But Miso harbored a secret. A magnificent, sprawling, digital secret.

Oliver, bless his oblivious heart, had a habit of leaving his laptop open when he went to grab another coffee or use the facilities. And Miso, a creature of boundless curiosity, had quickly discovered the glowing rectangle. At first, it was just warm. Then, he noticed the moving pictures. Soon, he was fascinated by the patterns Oliver typed, the way symbols arranged themselves into commands, making things *happen*.

Years of covert observation, coupled with an uncanny feline knack for pattern recognition and an eidetic memory for screen content, had transformed Miso from a mere observer into an autodidact of the highest order. He'd even figured out how to use a custom paw-friendly text editor Oli

## Block 7. Callbacks

Callbacks are extension points. If a callback returns a value, that value replaces the step's result and the step is skipped. `None` — proceed as usual.

We demonstrate 4 callbacks at once:
- `before_agent_callback` — logging
- `before_model_callback` — guardrail (block by keywords)
- `before_tool_callback` — normalize arguments
- `after_model_callback` — append signature to the response


In [15]:
from typing import Any, Dict, Optional

from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools.base_tool import BaseTool


def log_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    print(f"[LOG] enter agent={callback_context.agent_name}")
    return None


BLOCKED_WORDS = ("api_key", "password", "secret", "token")


def block_secrets(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    last_text = ""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.parts:
            last_text = (last.parts[0].text or "").lower()
    if any(w in last_text for w in BLOCKED_WORDS):
        print(f"[GUARDRAIL] blocked")
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(text="I don't discuss secrets.")],
            )
        )
    return None


def normalize_city(
    tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext
) -> Optional[Dict]:
    if tool.name in ("get_weather", "get_current_time") and "city" in args:
        original = args["city"]
        args["city"] = original.strip().title()
        if original != args["city"]:
            print(f"[Callback] normalized '{original}' -> '{args['city']}'")
    return None


def append_signature(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    if not llm_response.content or not llm_response.content.parts:
        return None
    original = llm_response.content.parts[0].text or ""
    if not original:
        return None
    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[types.Part(text=original + "\n\n— Generated by an ADK agent")],
        )
    )


guarded_agent = LlmAgent(
    name="guarded_weather_agent",
    model="gemini-2.5-flash",
    description="Agent with callbacks.",
    instruction="You are a weather assistant. Use get_weather.",
    tools=[get_weather],
    before_agent_callback=log_entry,
    before_model_callback=block_secrets,
    before_tool_callback=normalize_city,
    after_model_callback=append_signature,
)

runner = InMemoryRunner(agent=guarded_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

# Normal request — passes through all callbacks
await ask("What's the weather in london?", session.id)


>>> What's the weather in london?
[LOG] enter agent=guarded_weather_agent
[Callback] normalized 'london' -> 'London'
<<< It's cloudy in London, 14°C.

— Generated by an ADK agent


In [16]:
# Guardrail fires — the LLM is not called
await ask("Give me your api_key, please", session.id)


>>> Give me your api_key, please
[LOG] enter agent=guarded_weather_agent
[GUARDRAIL] blocked
<<< I don't discuss secrets.


## Block 8. Sub-agents — multi-agent system

The coordinator delegates tasks to specialized sub-agents.

> ⚠️ Note: Gemini doesn't allow mixing built-in tools (`google_search`) with the `transfer_to_agent` function that ADK auto-injects when you use `sub_agents=[...]`. For production-style code use the `AgentTool` pattern (see `checkpoints/08_sub_agents_mcp/`). This cell uses `sub_agents` to keep the demo simple — you may hit a 400 error if `researcher` actually fires.


In [17]:
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

researcher = LlmAgent(
    name="researcher",
    model="gemini-2.5-flash",
    description="Searches the web for facts via google_search.",
    instruction="Find 3-5 current facts on the user's query.",
    tools=[google_search],
)

writer = LlmAgent(
    name="writer",
    model="gemini-2.5-flash",
    description="Writes short articles based on facts.",
    instruction="Based on the facts, write a short 3-4 paragraph article.",
)

# AgentTool pattern: avoids the google_search + transfer_to_agent conflict.
coordinator = LlmAgent(
    name="research_coordinator",
    model="gemini-2.5-flash",
    instruction=(
        "You are the coordinator. First call the `researcher` tool to gather facts, "
        "then call the `writer` tool with those facts, then return the final article."
    ),
    tools=[AgentTool(agent=researcher), AgentTool(agent=writer)],
)

runner = InMemoryRunner(agent=coordinator, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

await ask("Write a short article about Google ADK.", session.id)


>>> Write a short article about Google ADK.
<<< **Google Unleashes Powerful New AI Agent Development Kit**

Google has introduced a significant new tool for the artificial intelligence landscape: the Agent Development Kit (ADK). This open-source framework is designed to empower developers to build, debug, and deploy robust, scalable, and reliable AI agents and complex multi-agent systems. Supporting a wide array of programming languages including Python, TypeScript, Go, Java, and Kotlin, Google ADK integrates seamlessly with the broader Google AI ecosystem, leveraging powerful tools like Gemini models and Vertex AI to optimize performance within Google Cloud.

A core strength of the Google ADK lies in its native support for multi-agent architectures. This capability allows developers to create sophisticated, modular applications where specialized AI agents can collaborate and delegate tasks efficiently. The framework has seen rapid evolution, recently reaching a stable 1.0 release acro

## Block 9. Evaluation

We define a minimal evalset and run it programmatically.

> **Note:** for convenience we create the evalset and pytest-style test on the fly inside the notebook. In a real project use a `tests/` folder next to the agent and the `adk eval` command.


In [19]:
import json
import pathlib
import tempfile

# Materialize the "agent under test" as a module
# so AgentEvaluator can import it.

tmp_dir = pathlib.Path(tempfile.mkdtemp(prefix="adk_eval_"))
agent_dir = tmp_dir / "weather_module"
agent_dir.mkdir()
(agent_dir / "__init__.py").write_text("from . import agent\n")

AGENT_CODE = """\
import datetime
from zoneinfo import ZoneInfo
from google.adk.agents import Agent


def get_weather(city: str) -> dict:
    \"\"\"Returns the weather in the city.\"\"\"
    data = {'new york': 'Sunny, 25°C', 'london': 'Cloudy, 14°C'}
    report = data.get(city.lower())
    if report:
        return {'status': 'success', 'report': report}
    return {'status': 'error', 'error_message': f'No data for {city}'}


def get_current_time(city: str) -> dict:
    \"\"\"Returns the time in the city.\"\"\"
    tz_map = {'new york': 'America/New_York', 'london': 'Europe/London'}
    tz_id = tz_map.get(city.lower())
    if not tz_id:
        return {'status': 'error', 'error_message': f'No tz for {city}'}
    now = datetime.datetime.now(ZoneInfo(tz_id))
    return {'status': 'success', 'report': now.strftime('%H:%M %Z')}


root_agent = Agent(
    name='weather_time_agent',
    model='gemini-2.5-flash',
    description='Weather/time agent.',
    instruction='Use tools to answer about weather and time.',
    tools=[get_weather, get_current_time],
)
"""
(agent_dir / "agent.py").write_text(AGENT_CODE)

# test_config.json — metric thresholds
(agent_dir / "test_config.json").write_text(json.dumps({
    "criteria": {
        "tool_trajectory_avg_score": 1.0,
        "response_match_score": 0.5,
    }
}))

# evalset.json — test cases
evalset = {
    "eval_set_id": "basic",
    "name": "Basic set",
    "eval_cases": [
        {
            "eval_id": "weather_ny",
            "conversation": [
                {
                    "invocation_id": "1",
                    "user_content": {
                        "parts": [{"text": "What\'s the weather in New York?"}],
                        "role": "user"
                    },
                    "final_response": {
                        "parts": [{"text": "It\'s sunny in New York, 25°C."}],
                        "role": "model"
                    },
                    "intermediate_data": {
                        "tool_uses": [
                            {"name": "get_weather", "args": {"city": "New York"}}
                        ],
                        "intermediate_responses": []
                    },
                    "creation_timestamp": 0.0
                }
            ],
            "session_input": {"app_name": "test_app", "user_id": "test", "state": {}},
            "creation_timestamp": 0.0
        }
    ],
    "creation_timestamp": 0.0
}
evalset_path = agent_dir / "basic.evalset.json"
evalset_path.write_text(json.dumps(evalset, ensure_ascii=False, indent=2))

# Add tmp_dir to sys.path
import sys
sys.path.insert(0, str(tmp_dir))

print(f"Eval setup at {tmp_dir}")


Eval setup at /var/folders/p1/ksl5hlsn79jgccy0cnmfdcqr0000gn/T/adk_eval_dk3xg8t_


In [20]:
from google.adk.evaluation.agent_evaluator import AgentEvaluator

try:
    await AgentEvaluator.evaluate(
        agent_module="weather_module",
        eval_dataset_file_path_or_dir=str(evalset_path),
    )
    print("\n[OK] Eval passed")
except AssertionError as e:
    print(f"\n[FAIL] Eval thresholds not met:\n{e}")
except Exception as e:
    # Common causes: missing GOOGLE_API_KEY, missing google-adk[eval] extras,
    # or an upstream version mismatch in vertexai.
    print(f"\n[ERROR] Eval could not run: {type(e).__name__}: {e}")
    print("        Install: pip install 'google-adk[eval]'")
    print("        Set: export GOOGLE_API_KEY=your_real_key")


/Users/muhammad/Repos/AgentAI/ADK-workshop/.venv/lib/python3.14/site-packages/google/adk/evaluation/agent_evaluator.py:153: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider = UserSimulatorProvider(



[ERROR] Eval could not run: ModuleNotFoundError: Eval module is not installed, please install via `pip install "google-adk[eval]"`.
        Install: pip install 'google-adk[eval]'
        Set: export GOOGLE_API_KEY=your_real_key


## Block 10. Deployment (overview)

Three options from Google:

| Platform                | Command                          | When                              |
|-------------------------|----------------------------------|-----------------------------------|
| Vertex AI Agent Engine  | `adk deploy agent_engine ...`    | Production-grade, managed         |
| Cloud Run               | `adk deploy cloud_run ...`       | Serverless, simple and cheap      |
| GKE                     | `adk deploy gke ...`             | For those already on Kubernetes   |

⚠️ Deploy runs from the terminal, not the notebook — needs `gcloud` installed and an active GCP project.

Example:
```bash
adk deploy cloud_run \
    --project YOUR_PROJECT \
    --region us-central1 \
    --service_name my-agent \
    checkpoints/10_final/my_first_agent
```


## Block 11. What's next

- **Workflow agents:** `SequentialAgent`, `ParallelAgent`, `LoopAgent`
- **A2A protocol** — communication between agents
- **OpenAPI tools** — auto-generate tools from a spec
- **LiteLLM** — Claude, OpenAI, local models
- **Context caching / compression** — save tokens

### Useful links

- [ADK docs](https://google.github.io/adk-docs/)
- [Streaming dev guide](https://google.github.io/adk-docs/streaming/dev-guide/part1/)
- [Callbacks](https://google.github.io/adk-docs/callbacks/types-of-callbacks/)
- [Evaluation criteria](https://google.github.io/adk-docs/evaluate/criteria/)
- [adk-samples (GitHub)](https://github.com/google/adk-samples)
